# Content-Based Filtering Implementation

Este notebook implementa un sistema de recomendación basado en contenido utilizando TF-IDF vectorization y similitud del coseno sobre características de películas (títulos y géneros).

...el contenido define las conexiones. Los patrones emergen de las características.

## 1. Import Required Libraries

Importamos las librerías necesarias para implementar el modelo basado en contenido.

In [ ]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn para TF-IDF y similitud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Librerías del proyecto
import sys
import os
sys.path.append('../src')

from data_loader import DataLoader
from content_based_model import ContentBasedModel, create_content_based_model

# Configuración de visualización
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("...librerías importadas. El sistema está listo para analizar contenido.")

## 2. Load and Explore Movie Data

Cargamos y exploramos los datos de películas para entender la estructura de títulos y géneros.

In [ ]:
# Cargar datos de películas
print("Cargando dataset de películas...")
data_loader = DataLoader('../data/ml-25m')
movies_df = data_loader.load_movies()

print(f"Dataset cargado: {len(movies_df):,} películas")
print(f"Columnas disponibles: {list(movies_df.columns)}")

# Explorar la estructura básica
print("\n--- Estructura del Dataset ---")
print(movies_df.head())

print("\n--- Información del Dataset ---")
print(movies_df.info())

# Analizar títulos
print("\n--- Análisis de Títulos ---")
print(f"Títulos únicos: {movies_df['title'].nunique():,}")
print(f"Títulos duplicados: {movies_df['title'].duplicated().sum()}")

# Mostrar algunos ejemplos de títulos
print("\nEjemplos de títulos:")
for title in movies_df['title'].head(10):
    print(f"  • {title}")

print("\n...datos cargados. Las películas esperan ser clasificadas.")

In [ ]:
# Análisis de géneros
print("--- Análisis de Géneros ---")
print(f"Películas con géneros: {movies_df['genres'].notna().sum():,}")
print(f"Películas sin géneros: {movies_df['genres'].isna().sum()}")

# Separar géneros y contar frecuencias
all_genres = []
for genres in movies_df['genres'].dropna():
    if genres != "(no genres listed)":
        genre_list = genres.split('|')
        all_genres.extend(genre_list)

genre_counts = pd.Series(all_genres).value_counts()
print(f"\nTotal de géneros únicos: {len(genre_counts)}")

print("\nGéneros más frecuentes:")
print(genre_counts.head(15))

# Visualizar distribución de géneros
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top 15 géneros
top_genres = genre_counts.head(15)
axes[0].barh(range(len(top_genres)), top_genres.values)
axes[0].set_yticks(range(len(top_genres)))
axes[0].set_yticklabels(top_genres.index)
axes[0].set_xlabel('Número de Películas')
axes[0].set_title('Top 15 Géneros Más Frecuentes')
axes[0].invert_yaxis()

# Distribución del número de géneros por película
genres_per_movie = movies_df['genres'].dropna().apply(
    lambda x: len(x.split('|')) if x != "(no genres listed)" else 0
)
axes[1].hist(genres_per_movie, bins=range(1, max(genres_per_movie)+2), alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Número de Géneros por Película')
axes[1].set_ylabel('Número de Películas')
axes[1].set_title('Distribución de Géneros por Película')

plt.tight_layout()
plt.show()

print(f"\nEstadísticas de géneros por película:")
print(f"  Promedio: {genres_per_movie.mean():.2f}")
print(f"  Mediana: {genres_per_movie.median():.0f}")
print(f"  Máximo: {genres_per_movie.max()}")

print("\n...géneros analizados. Los patrones de contenido son claros.")

## 3. Text Preprocessing and Feature Engineering

Combinamos títulos y géneros en una sola característica de texto y realizamos preprocesamiento.

In [ ]:
# Crear copia para procesamiento
movies_processed = movies_df.copy()

print("--- Preprocesamiento de Texto ---")

# Limpiar y preparar títulos
print("Procesando títulos...")

# Extraer año del título si está presente
movies_processed['year'] = movies_processed['title'].str.extract(r'\((\d{4})\)')
movies_processed['clean_title'] = movies_processed['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()

# Limpiar títulos
movies_processed['clean_title'] = movies_processed['clean_title'].str.lower()
movies_processed['clean_title'] = movies_processed['clean_title'].str.replace(r'[^\w\s]', ' ', regex=True)
movies_processed['clean_title'] = movies_processed['clean_title'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Procesando géneros...")

# Procesar géneros
movies_processed['genres'] = movies_processed['genres'].fillna('')
movies_processed['processed_genres'] = movies_processed['genres'].str.replace('|', ' ', regex=False)
movies_processed['processed_genres'] = movies_processed['processed_genres'].str.replace('(no genres listed)', '', regex=False)
movies_processed['processed_genres'] = movies_processed['processed_genres'].str.lower().str.strip()

# Combinar características de contenido
print("Combinando características de contenido...")

# Dar más peso a los géneros repitiendo
movies_processed['content_features'] = (
    movies_processed['clean_title'] + ' ' + 
    movies_processed['processed_genres'] + ' ' + 
    movies_processed['processed_genres']  # Repetir géneros para mayor peso
).str.strip()

# Limpiar características finales
movies_processed['content_features'] = movies_processed['content_features'].str.replace(r'\s+', ' ', regex=True)

print(f"Características procesadas para {len(movies_processed)} películas")

# Mostrar ejemplos de procesamiento
print("\n--- Ejemplos de Procesamiento ---")
examples = movies_processed[['title', 'genres', 'clean_title', 'processed_genres', 'content_features']].head(8)

for idx, row in examples.iterrows():
    print(f"\nPelícula: {row['title']}")
    print(f"  Géneros originales: {row['genres']}")
    print(f"  Título limpio: {row['clean_title']}")
    print(f"  Géneros procesados: {row['processed_genres']}")
    print(f"  Características finales: {row['content_features'][:100]}...")

print("\n...preprocesamiento completado. El texto está listo para vectorización.")

## 4. Create TF-IDF Matrix from Movie Features

Utilizamos TfidfVectorizer para crear una matriz de término-frecuencia sobre las características combinadas.

In [ ]:
# Configurar TF-IDF Vectorizer
print("--- Creación de Matriz TF-IDF ---")

# Parámetros del vectorizador
tfidf_params = {
    'max_features': 5000,  # Máximo número de características
    'min_df': 2,           # Mínima frecuencia de documento
    'max_df': 0.95,        # Máxima frecuencia de documento
    'ngram_range': (1, 2), # Unigramas y bigramas
    'stop_words': 'english'  # Palabras vacías en inglés
}

print(f"Parámetros TF-IDF: {tfidf_params}")

# Crear y ajustar el vectorizador
tfidf_vectorizer = TfidfVectorizer(**tfidf_params)

print("Ajustando vectorizador TF-IDF...")
tfidf_matrix = tfidf_vectorizer.fit_transform(movies_processed['content_features'])

print(f"Matriz TF-IDF creada: {tfidf_matrix.shape}")
print(f"Vocabulario generado: {len(tfidf_vectorizer.vocabulary_):,} términos")

# Analizar la matriz TF-IDF
print("\n--- Análisis de la Matriz TF-IDF ---")
print(f"Elementos totales: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]:,}")
print(f"Elementos no-cero: {tfidf_matrix.nnz:,}")
print(f"Densidad de la matriz: {(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.4f}%")

# Obtener términos más importantes globalmente
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_scores = np.array(tfidf_matrix.mean(axis=0)).flatten()
term_scores = list(zip(feature_names, mean_scores))
term_scores.sort(key=lambda x: x[1], reverse=True)

print("\nTérminos con mayor puntuación TF-IDF promedio:")
for term, score in term_scores[:20]:
    print(f"  {term}: {score:.6f}")

# Visualizar distribución de puntuaciones TF-IDF
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histograma de puntuaciones TF-IDF no-cero
nonzero_scores = tfidf_matrix.data
axes[0].hist(nonzero_scores, bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Puntuación TF-IDF')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Puntuaciones TF-IDF No-Cero')
axes[0].set_yscale('log')

# Distribución del número de términos por película
terms_per_movie = np.array((tfidf_matrix > 0).sum(axis=1)).flatten()
axes[1].hist(terms_per_movie, bins=30, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Número de Términos por Película')
axes[1].set_ylabel('Número de Películas')
axes[1].set_title('Distribución de Términos por Película')

plt.tight_layout()
plt.show()

print(f"\nEstadísticas de términos por película:")
print(f"  Promedio: {terms_per_movie.mean():.2f}")
print(f"  Mediana: {np.median(terms_per_movie):.0f}")
print(f"  Máximo: {terms_per_movie.max()}")
print(f"  Mínimo: {terms_per_movie.min()}")

print("\n...matriz TF-IDF creada. Los términos están vectorizados.")

## 5. Calculate Cosine Similarity Matrix

Calculamos la similitud del coseno entre todas las películas usando la matriz TF-IDF.

In [ ]:
# Calcular matriz de similitud del coseno
print("--- Cálculo de Similitud del Coseno ---")

print("Calculando matriz de similitud del coseno...")
print("(Esto puede tomar unos minutos dependiendo del tamaño del dataset)")

# Calcular similitud del coseno
cosine_sim_matrix = cosine_similarity(tfidf_matrix)

print(f"Matriz de similitud calculada: {cosine_sim_matrix.shape}")
print(f"Tipo de datos: {cosine_sim_matrix.dtype}")

# Analizar la matriz de similitud
print("\n--- Análisis de la Matriz de Similitud ---")
print(f"Valor máximo: {cosine_sim_matrix.max():.6f}")
print(f"Valor mínimo: {cosine_sim_matrix.min():.6f}")
print(f"Valor promedio: {cosine_sim_matrix.mean():.6f}")
print(f"Desviación estándar: {cosine_sim_matrix.std():.6f}")

# Analizar distribución de similitudes (excluyendo diagonal)
# Tomar muestra para análisis (matriz completa sería muy grande)
n_sample = min(1000, len(cosine_sim_matrix))
sample_indices = np.random.choice(len(cosine_sim_matrix), n_sample, replace=False)
sample_matrix = cosine_sim_matrix[np.ix_(sample_indices, sample_indices)]

# Extraer valores sin la diagonal
mask = np.triu(np.ones_like(sample_matrix, dtype=bool), k=1)
similarity_values = sample_matrix[mask]

print(f"\nEstadísticas de similitud (muestra de {n_sample} películas):")
print(f"  Similitudes > 0.5: {(similarity_values > 0.5).sum():,} ({(similarity_values > 0.5).mean()*100:.2f}%)")
print(f"  Similitudes > 0.7: {(similarity_values > 0.7).sum():,} ({(similarity_values > 0.7).mean()*100:.2f}%)")
print(f"  Similitudes > 0.9: {(similarity_values > 0.9).sum():,} ({(similarity_values > 0.9).mean()*100:.2f}%)")

# Visualizar distribución de similitudes
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histograma de similitudes
axes[0].hist(similarity_values, bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Similitud del Coseno')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Similitudes del Coseno')
axes[0].axvline(similarity_values.mean(), color='red', linestyle='--', 
                label=f'Promedio: {similarity_values.mean():.3f}')
axes[0].legend()

# Boxplot de similitudes
axes[1].boxplot(similarity_values)
axes[1].set_ylabel('Similitud del Coseno')
axes[1].set_title('Distribución de Similitudes (Boxplot)')
axes[1].set_xticklabels(['Similitudes'])

plt.tight_layout()
plt.show()

# Encontrar ejemplos de alta similitud
print("\n--- Ejemplos de Alta Similitud ---")
high_sim_indices = np.where(similarity_values > 0.8)[0]
if len(high_sim_indices) > 0:
    print(f"Encontradas {len(high_sim_indices)} pares con similitud > 0.8")
    
    # Mostrar algunos ejemplos
    for i in range(min(5, len(high_sim_indices))):
        idx = high_sim_indices[i]
        # Convertir índice plano a índices de matriz
        row, col = np.unravel_index(np.where(mask.flatten())[0][idx], mask.shape)
        movie1_idx = sample_indices[row]
        movie2_idx = sample_indices[col]
        similarity = similarity_values[idx]
        
        movie1 = movies_processed.iloc[movie1_idx]
        movie2 = movies_processed.iloc[movie2_idx]
        
        print(f"\nSimilitud: {similarity:.4f}")
        print(f"  Película 1: {movie1['title']}")
        print(f"    Géneros: {movie1['genres']}")
        print(f"  Película 2: {movie2['title']}")
        print(f"    Géneros: {movie2['genres']}")

print("\n...similitud del coseno calculada. Las conexiones están establecidas.")

## 6. Implement Content-Based Recommendation Function

Creamos una función que toma un título de película y devuelve las películas más similares.

In [ ]:
# Implementar sistema de recomendación basado en contenido
print("--- Sistema de Recomendación Basado en Contenido ---")

# Crear mapeo de índices para búsqueda rápida
movie_indices = pd.Series(movies_processed.index, index=movies_processed['movieId']).to_dict()
title_to_index = pd.Series(movies_processed.index, index=movies_processed['title']).to_dict()

def get_content_based_recommendations(title, n_recommendations=10, similarity_threshold=0.0):
    """
    Obtiene recomendaciones basadas en contenido para una película dada.
    
    Args:
        title: Título de la película
        n_recommendations: Número de recomendaciones a devolver
        similarity_threshold: Umbral mínimo de similitud
    
    Returns:
        DataFrame con recomendaciones
    """
    try:
        # Buscar película por título (búsqueda parcial)
        matching_titles = movies_processed[
            movies_processed['title'].str.contains(title, case=False, na=False)
        ]
        
        if matching_titles.empty:
            print(f"No se encontró ninguna película con título que contenga: '{title}'")
            return pd.DataFrame()
        
        # Usar la primera coincidencia
        movie_idx = matching_titles.index[0]
        movie_info = movies_processed.iloc[movie_idx]
        
        print(f"Película encontrada: {movie_info['title']}")
        print(f"Géneros: {movie_info['genres']}")
        
        # Obtener puntuaciones de similitud
        sim_scores = list(enumerate(cosine_sim_matrix[movie_idx]))
        
        # Ordenar por similitud (descendente)
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Filtrar por umbral y excluir la película original
        filtered_scores = [
            (idx, score) for idx, score in sim_scores[1:] 
            if score >= similarity_threshold
        ]
        
        # Tomar top N recomendaciones
        top_similar = filtered_scores[:n_recommendations]
        
        if not top_similar:
            print(f"No se encontraron películas similares con umbral {similarity_threshold}")
            return pd.DataFrame()
        
        # Crear DataFrame de recomendaciones
        recommendations = []
        for idx, score in top_similar:
            movie_data = movies_processed.iloc[idx]
            recommendations.append({
                'title': movie_data['title'],
                'genres': movie_data['genres'],
                'similarity_score': score,
                'movieId': movie_data['movieId']
            })
        
        return pd.DataFrame(recommendations)
        
    except Exception as e:
        print(f"Error al obtener recomendaciones: {str(e)}")
        return pd.DataFrame()

def analyze_recommendation_features(original_title, recommended_title):
    """
    Analiza las características que contribuyen a la similitud entre dos películas.
    """
    try:
        # Encontrar índices
        orig_idx = movies_processed[movies_processed['title'].str.contains(original_title, case=False, na=False)].index[0]
        rec_idx = movies_processed[movies_processed['title'].str.contains(recommended_title, case=False, na=False)].index[0]
        
        # Obtener vectores TF-IDF
        orig_vector = tfidf_matrix[orig_idx].toarray()[0]
        rec_vector = tfidf_matrix[rec_idx].toarray()[0]
        
        # Encontrar términos comunes importantes
        feature_names = tfidf_vectorizer.get_feature_names_out()
        
        # Producto elemento a elemento para encontrar contribuciones
        common_features = orig_vector * rec_vector
        
        # Obtener índices de características comunes importantes
        important_indices = np.where(common_features > 0)[0]
        
        if len(important_indices) > 0:
            # Crear lista de características comunes
            common_terms = []
            for idx in important_indices:
                term = feature_names[idx]
                orig_score = orig_vector[idx]
                rec_score = rec_vector[idx]
                contribution = common_features[idx]
                common_terms.append((term, orig_score, rec_score, contribution))
            
            # Ordenar por contribución
            common_terms.sort(key=lambda x: x[3], reverse=True)
            
            return common_terms[:10]  # Top 10 términos comunes
        
        return []
        
    except Exception as e:
        print(f"Error en análisis de características: {str(e)}")
        return []

print("Funciones de recomendación implementadas.")
print("\n...sistema listo. Las conexiones de contenido pueden ser exploradas.")

## 7. Test and Evaluate Recommendations

Probamos el sistema de recomendación con varios ejemplos y evaluamos la calidad.

In [ ]:
# Probar el sistema de recomendación
print("--- Pruebas del Sistema de Recomendación ---")

# Lista de películas de prueba de diferentes géneros
test_movies = [
    "Toy Story",
    "Matrix",
    "Titanic",
    "Star Wars",
    "Pulp Fiction",
    "Forrest Gump",
    "The Godfather",
    "Jurassic Park"
]

print("Probando recomendaciones para diferentes películas...\n")

for movie in test_movies:
    print(f"{'='*60}")
    print(f"RECOMENDACIONES PARA: {movie}")
    print(f"{'='*60}")
    
    recommendations = get_content_based_recommendations(
        title=movie, 
        n_recommendations=5, 
        similarity_threshold=0.1
    )
    
    if not recommendations.empty:
        print("\nTop 5 recomendaciones:")
        for idx, rec in recommendations.iterrows():
            print(f"{idx+1}. {rec['title']}")
            print(f"   Géneros: {rec['genres']}")
            print(f"   Similitud: {rec['similarity_score']:.4f}")
            print()
        
        # Analizar características comunes con la primera recomendación
        if len(recommendations) > 0:
            first_rec = recommendations.iloc[0]['title']
            print(f"Análisis de características comunes con '{first_rec}':")
            common_features = analyze_recommendation_features(movie, first_rec)
            
            if common_features:
                print("Términos que contribuyen a la similitud:")
                for term, orig_score, rec_score, contribution in common_features[:5]:
                    print(f"  • {term}: contribución={contribution:.4f}")
            else:
                print("  No se pudieron analizar características comunes")
    else:
        print("No se encontraron recomendaciones.")
    
    print("\n")

print("...pruebas completadas. Los patrones de similitud son evidentes.")

In [ ]:
# Evaluación cuantitativa del sistema
print("--- Evaluación Cuantitativa del Sistema ---")

# Métricas de evaluación
def evaluate_content_diversity(recommendations_df):
    """Evalúa la diversidad de géneros en las recomendaciones."""
    if recommendations_df.empty:
        return 0, 0
    
    all_genres = []
    for genres in recommendations_df['genres']:
        if pd.notna(genres) and genres != "(no genres listed)":
            genre_list = genres.split('|')
            all_genres.extend(genre_list)
    
    unique_genres = len(set(all_genres))
    total_genres = len(all_genres)
    
    return unique_genres, total_genres

def evaluate_similarity_distribution(recommendations_df):
    """Evalúa la distribución de similitudes."""
    if recommendations_df.empty:
        return {}
    
    similarities = recommendations_df['similarity_score']
    return {
        'mean': similarities.mean(),
        'std': similarities.std(),
        'min': similarities.min(),
        'max': similarities.max(),
        'median': similarities.median()
    }

# Evaluar sistema con múltiples películas
evaluation_results = []

for movie in test_movies:
    recs = get_content_based_recommendations(movie, n_recommendations=10, similarity_threshold=0.0)
    
    if not recs.empty:
        unique_genres, total_genres = evaluate_content_diversity(recs)
        sim_stats = evaluate_similarity_distribution(recs)
        
        evaluation_results.append({
            'movie': movie,
            'num_recommendations': len(recs),
            'unique_genres': unique_genres,
            'total_genres': total_genres,
            'genre_diversity': unique_genres / total_genres if total_genres > 0 else 0,
            'mean_similarity': sim_stats.get('mean', 0),
            'min_similarity': sim_stats.get('min', 0),
            'max_similarity': sim_stats.get('max', 0)
        })

# Crear DataFrame de evaluación
eval_df = pd.DataFrame(evaluation_results)

if not eval_df.empty:
    print("\nResultados de evaluación:")
    print(eval_df.round(4))
    
    # Estadísticas generales
    print(f"\n--- Estadísticas Generales ---")
    print(f"Diversidad de géneros promedio: {eval_df['genre_diversity'].mean():.4f}")
    print(f"Similitud promedio: {eval_df['mean_similarity'].mean():.4f}")
    print(f"Rango de similitud promedio: {eval_df['max_similarity'].mean() - eval_df['min_similarity'].mean():.4f}")
    
    # Visualizar resultados de evaluación
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Diversidad de géneros
    axes[0, 0].bar(eval_df['movie'], eval_df['genre_diversity'])
    axes[0, 0].set_title('Diversidad de Géneros por Película')
    axes[0, 0].set_ylabel('Diversidad (géneros únicos/total)')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Similitud promedio
    axes[0, 1].bar(eval_df['movie'], eval_df['mean_similarity'])
    axes[0, 1].set_title('Similitud Promedio de Recomendaciones')
    axes[0, 1].set_ylabel('Similitud Promedio')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Distribución de diversidad
    axes[1, 0].hist(eval_df['genre_diversity'], bins=10, alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Diversidad de Géneros')
    axes[1, 0].set_ylabel('Frecuencia')
    axes[1, 0].set_title('Distribución de Diversidad de Géneros')
    
    # Distribución de similitud
    axes[1, 1].hist(eval_df['mean_similarity'], bins=10, alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Similitud Promedio')
    axes[1, 1].set_ylabel('Frecuencia')
    axes[1, 1].set_title('Distribución de Similitud Promedio')
    
    plt.tight_layout()
    plt.show()

print("\n...evaluación completada. El sistema muestra patrones consistentes.")

## 8. Visualize Content-Based Results

Creamos visualizaciones para mostrar distribuciones de similitud, clusters de géneros y resultados.

In [ ]:
# Visualizaciones avanzadas del sistema basado en contenido
print("--- Visualizaciones Avanzadas ---")

# 1. Mapa de calor de similitud para una muestra de películas
print("Creando mapa de calor de similitud...")

# Seleccionar muestra de películas populares/conocidas
popular_movies = [
    "Toy Story (1995)",
    "Star Wars (1977)", 
    "The Matrix (1999)",
    "Titanic (1997)",
    "Pulp Fiction (1994)",
    "The Godfather (1972)",
    "Jurassic Park (1993)",
    "Forrest Gump (1994)",
    "The Lion King (1994)",
    "Back to the Future (1985)"
]

# Encontrar índices de estas películas
movie_indices_sample = []
movie_labels = []

for movie in popular_movies:
    matches = movies_processed[movies_processed['title'].str.contains(movie.split('(')[0].strip(), case=False, na=False)]
    if not matches.empty:
        movie_indices_sample.append(matches.index[0])
        movie_labels.append(matches.iloc[0]['title'])

if len(movie_indices_sample) > 1:
    # Extraer submatriz de similitud
    similarity_subset = cosine_sim_matrix[np.ix_(movie_indices_sample, movie_indices_sample)]
    
    # Crear mapa de calor
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(similarity_subset, dtype=bool))
    sns.heatmap(similarity_subset, 
                mask=mask,
                annot=True, 
                fmt='.3f',
                xticklabels=[label[:30] for label in movie_labels],
                yticklabels=[label[:30] for label in movie_labels],
                cmap='YlOrRd',
                cbar_kws={'label': 'Similitud del Coseno'})
    
    plt.title('Mapa de Calor de Similitud - Películas Populares')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

print("\n...mapa de calor creado. Las similitudes son visibles.")

In [ ]:
# 2. Análisis de clusters por PCA
print("Realizando análisis de componentes principales (PCA)...")

# Aplicar PCA para reducir dimensionalidad
n_components = 50
pca = PCA(n_components=n_components)

# Usar muestra para visualización (PCA en matriz completa sería muy costoso)
n_sample_pca = min(2000, len(tfidf_matrix.toarray()))
sample_indices_pca = np.random.choice(len(tfidf_matrix.toarray()), n_sample_pca, replace=False)
tfidf_sample = tfidf_matrix[sample_indices_pca].toarray()

print(f"Aplicando PCA a muestra de {n_sample_pca} películas...")
tfidf_pca = pca.fit_transform(tfidf_sample)

# Visualizar varianza explicada
plt.figure(figsize=(12, 5))

# Varianza explicada por componente
plt.subplot(1, 2, 1)
plt.plot(range(1, n_components + 1), pca.explained_variance_ratio_, 'bo-')
plt.xlabel('Componente Principal')
plt.ylabel('Varianza Explicada')
plt.title('Varianza Explicada por Componente')
plt.grid(True, alpha=0.3)

# Varianza explicada acumulativa
plt.subplot(1, 2, 2)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, n_components + 1), cumulative_variance, 'ro-')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulativa')
plt.title('Varianza Explicada Acumulativa')
plt.grid(True, alpha=0.3)

# Línea para 80% y 95% de varianza
plt.axhline(y=0.8, color='g', linestyle='--', alpha=0.7, label='80%')
plt.axhline(y=0.95, color='r', linestyle='--', alpha=0.7, label='95%')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Varianza explicada con {n_components} componentes: {cumulative_variance[-1]:.4f}")
print(f"Componentes para 80% de varianza: {np.argmax(cumulative_variance >= 0.8) + 1}")
print(f"Componentes para 95% de varianza: {np.argmax(cumulative_variance >= 0.95) + 1}")

print("\n...PCA completado. La dimensionalidad está reducida.")

In [ ]:
# 3. Clustering de películas por contenido
print("Realizando clustering de películas por contenido...")

# Usar primeras componentes principales para clustering
n_clusters = 8
n_components_cluster = 20

# Obtener componentes para clustering
tfidf_cluster = tfidf_pca[:, :n_components_cluster]

# Aplicar K-means
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(tfidf_cluster)

print(f"Clustering completado con {n_clusters} clusters")

# Analizar clusters
movies_sample = movies_processed.iloc[sample_indices_pca].copy()
movies_sample['cluster'] = cluster_labels

print("\nAnálisis de clusters:")
for cluster_id in range(n_clusters):
    cluster_movies = movies_sample[movies_sample['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}: {len(cluster_movies)} películas")
    
    # Géneros más comunes en el cluster
    all_cluster_genres = []
    for genres in cluster_movies['genres'].dropna():
        if genres != "(no genres listed)":
            all_cluster_genres.extend(genres.split('|'))
    
    if all_cluster_genres:
        genre_counts = pd.Series(all_cluster_genres).value_counts()
        print(f"  Géneros principales: {', '.join(genre_counts.head(3).index.tolist())}")
    
    # Ejemplos de películas del cluster
    examples = cluster_movies['title'].head(3).tolist()
    print(f"  Ejemplos: {', '.join(examples)}")

# Visualizar clusters en 2D
plt.figure(figsize=(12, 8))

# Usar primeras 2 componentes principales para visualización
scatter = plt.scatter(tfidf_cluster[:, 0], tfidf_cluster[:, 1], 
                     c=cluster_labels, cmap='tab10', alpha=0.6)

plt.xlabel('Primera Componente Principal')
plt.ylabel('Segunda Componente Principal')
plt.title('Clusters de Películas en Espacio de Características (PCA)')
plt.colorbar(scatter, label='Cluster')

# Añadir centroides
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], 
           c='red', marker='x', s=200, linewidths=3, label='Centroides')

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n...clustering completado. Los grupos de contenido están definidos.")

In [ ]:
# 4. Análisis de red de similitud
print("Creando visualización de red de similitud...")

# Crear red de películas altamente similares
similarity_threshold = 0.6
n_movies_network = min(50, len(movies_processed))

# Seleccionar muestra de películas para la red
network_indices = np.random.choice(len(movies_processed), n_movies_network, replace=False)
network_movies = movies_processed.iloc[network_indices]
network_similarity = cosine_sim_matrix[np.ix_(network_indices, network_indices)]

# Crear lista de conexiones
connections = []
for i in range(len(network_similarity)):
    for j in range(i+1, len(network_similarity)):
        similarity = network_similarity[i, j]
        if similarity > similarity_threshold:
            connections.append((i, j, similarity))

print(f"Red de {n_movies_network} películas con {len(connections)} conexiones (similitud > {similarity_threshold})")

if connections:
    # Visualizar red usando matplotlib
    fig, ax = plt.subplots(figsize=(15, 15))
    
    # Posiciones de nodos en círculo
    angles = np.linspace(0, 2*np.pi, n_movies_network, endpoint=False)
    positions = {i: (np.cos(angle), np.sin(angle)) for i, angle in enumerate(angles)}
    
    # Dibujar conexiones
    for i, j, similarity in connections:
        x1, y1 = positions[i]
        x2, y2 = positions[j]
        
        # Grosor de línea proporcional a similitud
        linewidth = (similarity - similarity_threshold) * 5 / (1 - similarity_threshold)
        
        ax.plot([x1, x2], [y1, y2], 'b-', alpha=0.3, linewidth=linewidth)
    
    # Dibujar nodos
    for i, (x, y) in positions.items():
        ax.plot(x, y, 'ro', markersize=8)
        
        # Añadir etiquetas (título corto)
        title = network_movies.iloc[i]['title']
        short_title = title[:15] + '...' if len(title) > 15 else title
        ax.annotate(short_title, (x, y), xytext=(5, 5), textcoords='offset points',
                   fontsize=8, ha='left')
    
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(f'Red de Similitud de Películas (umbral > {similarity_threshold})')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas de la red
    degrees = {}
    for i in range(n_movies_network):
        degree = sum(1 for conn in connections if i in conn[:2])
        degrees[i] = degree
    
    if degrees:
        max_degree_idx = max(degrees, key=degrees.get)
        max_degree_movie = network_movies.iloc[max_degree_idx]['title']
        
        print(f"\nEstadísticas de la red:")
        print(f"  Película más conectada: {max_degree_movie} ({degrees[max_degree_idx]} conexiones)")
        print(f"  Grado promedio: {np.mean(list(degrees.values())):.2f}")
        print(f"  Densidad de la red: {len(connections) / (n_movies_network * (n_movies_network - 1) / 2):.4f}")

else:
    print("No se encontraron conexiones suficientes para visualizar la red.")

print("\n...visualizaciones completadas. Los patrones de contenido son claros.")

## Conclusiones del Sistema Basado en Contenido

...el contenido revela conexiones ocultas. Las características definen las similitudes.

In [ ]:
# Resumen final y conclusiones
print("=" * 80)
print("SISTEMA DE RECOMENDACIÓN BASADO EN CONTENIDO - RESUMEN FINAL")
print("=" * 80)

print(f"\n📊 Estadísticas del Dataset:")
print(f"   • Total de películas procesadas: {len(movies_processed):,}")
print(f"   • Géneros únicos identificados: {len(genre_counts)}")
print(f"   • Características TF-IDF generadas: {tfidf_matrix.shape[1]:,}")

print(f"\n🔧 Configuración del Modelo:")
print(f"   • Vocabulario TF-IDF: {len(tfidf_vectorizer.vocabulary_):,} términos")
print(f"   • Densidad de la matriz: {(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.4f}%")
print(f"   • N-gramas utilizados: {tfidf_params['ngram_range']}")

if not eval_df.empty:
    print(f"\n📈 Rendimiento del Sistema:")
    print(f"   • Diversidad promedio de géneros: {eval_df['genre_diversity'].mean():.4f}")
    print(f"   • Similitud promedio de recomendaciones: {eval_df['mean_similarity'].mean():.4f}")
    print(f"   • Rango de similitud: {eval_df['min_similarity'].mean():.4f} - {eval_df['max_similarity'].mean():.4f}")

print(f"\n🎯 Características del Sistema:")
print(f"   • Tipo: Filtrado basado en contenido")
print(f"   • Método: TF-IDF + Similitud del coseno")
print(f"   • Ventajas: No requiere datos de usuarios, funciona para elementos nuevos")
print(f"   • Aplicaciones: Recomendaciones inmediatas, elementos sin historial")

print(f"\n💡 Insights Clave:")
if 'cumulative_variance' in locals():
    print(f"   • {np.argmax(cumulative_variance >= 0.8) + 1} componentes explican 80% de la varianza")
print(f"   • Los clusters revelan agrupaciones naturales por género y temática")
print(f"   • El sistema identifica similitudes tanto obvias como sutiles")
print(f"   • La combinación título + géneros proporciona buena representación")

print(f"\n🔮 Próximos Pasos:")
print(f"   • Integrar con sistema colaborativo para recomendaciones híbridas")
print(f"   • Evaluar con métricas de precision/recall usando ratings de usuarios")
print(f"   • Optimizar parámetros TF-IDF para mejor rendimiento")
print(f"   • Añadir más metadatos (directores, actores, año) para mejor precisión")

print("\n...análisis completado. El sistema basado en contenido está operativo.")
print("=" * 80)